# FactoryGuard AI — Exploratory Data Analysis
**Infotact Solutions | Cohort Zeta | Q4 2025**

This notebook is strictly for EDA, visualization, and feature prototyping.
Production code lives in `src/` as modular `.py` files.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/sensor_data.csv')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head(10)

## 1. Dataset Overview & Basic Statistics

In [ ]:
print('=== SHAPE ===')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
print()
print('=== DATA TYPES ===')
print(df.dtypes)
print()
print('=== NULL VALUES ===')
print(df.isnull().sum())
print()
print('=== BASIC STATISTICS ===')
df.describe()

## 2. Class Imbalance Analysis

In [ ]:
failure_counts = df['failure'].value_counts()
failure_pct = df['failure'].value_counts(normalize=True) * 100
print('=== CLASS DISTRIBUTION ===')
print(f'Normal  (0): {failure_counts[0]:,} samples ({failure_pct[0]:.1f}%)')
print(f'Failure (1): {failure_counts[1]:,} samples ({failure_pct[1]:.1f}%)')
print()
print('THIS IS WHY WE USE SMOTE - severe class imbalance!')

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(['Normal', 'Failure'], failure_counts.values, color=['#2196F3','#F44336'])
ax.set_title('Class Distribution - Failure vs Normal')
ax.set_ylabel('Count')
for i, v in enumerate(failure_counts.values):
    ax.text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../class_distribution.png', dpi=150)
print('Saved: class_distribution.png')

## 3. Correlation Matrix

In [ ]:
numeric_df = df.select_dtypes(include='number')
corr = numeric_df.corr()
print('Correlation Matrix:')
print(corr.round(3))

fig, ax = plt.subplots(figsize=(8,6))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
plt.colorbar(im)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=9, fontweight='bold')
ax.set_title('Sensor Correlation Matrix - FactoryGuard AI', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../correlation_matrix.png', dpi=150)
print('Saved: correlation_matrix.png')

## 4. Sensor Distribution Analysis

In [ ]:
sensor_cols = ['vibration', 'temperature', 'pressure']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {'Normal': '#2196F3', 'Failure': '#F44336'}

for i, col in enumerate(sensor_cols):
    for label, group in df.groupby('failure'):
        name = 'Failure' if label == 1 else 'Normal'
        axes[i].hist(group[col], bins=40, alpha=0.6, label=name, color=colors[name])
    axes[i].set_title(f'{col.capitalize()} Distribution')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].legend()

plt.suptitle('Sensor Readings: Normal vs Failure', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../sensor_distributions.png', dpi=150)
print('Saved: sensor_distributions.png')

## 5. Time Series View per Machine

In [ ]:
machine = df[df['machine_id'] == 'M001'].copy()
machine['timestamp'] = pd.to_datetime(machine['timestamp'])

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
for i, col in enumerate(['vibration', 'temperature', 'pressure']):
    axes[i].plot(machine['timestamp'], machine[col], linewidth=0.8, color='#1565C0')
    failures = machine[machine['failure'] == 1]
    axes[i].scatter(failures['timestamp'], failures[col], color='red', s=20, zorder=5, label='Failure')
    axes[i].set_ylabel(col.capitalize())
    axes[i].legend(loc='upper right')

axes[0].set_title('Machine M001 - Sensor Time Series (Red = Failure)', fontweight='bold')
plt.tight_layout()
plt.savefig('../timeseries_M001.png', dpi=150)
print('Saved: timeseries_M001.png')

## 6. SHAP Summary Plot Analysis

The SHAP summary plot shows which features most impact the model's predictions.
- Features sorted by importance (top = most important)
- Red dots = high feature value, Blue = low
- X-axis = SHAP value (impact on prediction)

In [ ]:
from IPython.display import Image
print('SHAP Summary Plot (generated by run_training.py):')
Image('../shap_summary_plot.png')

## 7. EDA Conclusions

**Key Findings:**
1. **Severe Class Imbalance**: ~2% failure rate — SMOTE is mandatory
2. **Temperature Correlation**: High temperature strongly correlates with failure
3. **Vibration Patterns**: Failure machines show higher vibration variance
4. **Pressure Stability**: Failures preceded by pressure instability
5. **Temporal Drift**: Rolling features are essential — single readings insufficient

**Design Decisions Validated:**
- Use F1/Recall NOT accuracy (imbalanced classes)
- SMOTE for minority class oversampling
- Rolling windows 1h/4h/8h to capture temporal trends
- XGBoost over linear models (non-linear sensor relationships)